# RBI/SEBI Regulation Copilot - RAG + AI agent, step by step

Ask a question about RBI/SEBI rules and get an answer **only from the official circulars**, with page citations.

| Part | What we build | RAG / agent idea |
|---|---|---|
| 1 | Download the official PDFs | Ingestion |
| 2 | Read every page into a table | Text extraction + metadata |
| 3 | Cut pages into chunks | Chunking |
| 4 | Turn chunks into numbers, search by meaning | Embeddings, vector search |
| 5 | Add keyword search, merge, re-rank | Hybrid search, RRF, cross-encoder |
| 6 | Write the answer with citations - 3 engines | Generation, graceful fallback, refusal threshold |
| 7 | Score it on questions with known answers | Evaluation |
| 8 | Let Claude choose tools in a loop | AI agent (tool use) |
| 9 | Save the index for the website | Deployment |

**How to run:** Runtime -> Change runtime type -> **T4 GPU** (optional, enables the free local model), then **Runtime -> Run all**.
**API key (optional):** add `ANTHROPIC_API_KEY` in the 🔑 Secrets panel. Without it everything still works.

## Part 1 - Setup and download
### Connect Google Drive
Everything is saved in one Drive folder so it survives restarts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os
BASE = "/content/drive/MyDrive/regcopilot"   # project folder
PDFS = f"{BASE}/pdfs"                        # downloaded circulars
OUT  = f"{BASE}/outputs"                     # chunks, vectors, results
os.makedirs(PDFS, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(PDFS)))

### Download the RBI Master Directions
We read RBI's list page, find each topic, skip amendments and ombudsman documents, and download the PDF like a browser would. Files already downloaded are skipped.

In [ ]:
import requests, re, html

LIST_PAGE = "https://www.rbi.org.in/Scripts/BS_ViewMasterDirections.aspx"
WANTED = {
    "Know Your Customer": "RBI_KYC_Master_Direction.pdf",
    "Prepaid Payment Instruments": "RBI_PPI_Master_Direction.pdf",
    "Digital Lending": "RBI_Digital_Lending_Directions.pdf",
    "Fraud Risk Management": "RBI_Fraud_Risk_Management.pdf",
    "Payment Aggregator": "RBI_Payment_Aggregators.pdf",
}
SKIP_WORDS = ["ombudsman", "amendment", "repeal", "withdrawal"]          # titles we don't want
pdf_link = re.compile(r"""https?://rbidocs\.rbi\.org\.in/[^"'\s>]+?\.pdf""", flags=re.I)

s = requests.Session()                                                   # keeps cookies like a browser tab
s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
                  "Accept": "application/pdf,text/html;q=0.9,*/*;q=0.8", "Referer": LIST_PAGE})
page = s.get(LIST_PAGE, timeout=60).text
print(len(page), "characters on RBI's list page")

In [ ]:
for topic, filename in WANTED.items():
    target = f"{PDFS}/{filename}"
    if os.path.exists(target):
        print("Already have", filename); continue

    found = None
    for m in re.finditer(re.escape(topic), page, flags=re.I):            # every mention of the topic
        title = re.sub(r"<[^>]+>|\s+", " ", page[m.start(): m.start() + 200]).strip()
        if any(w in title.lower() for w in SKIP_WORDS):
            continue
        pdf = pdf_link.search(page[m.start(): m.start() + 3000])         # first RBI PDF link after it
        if pdf:
            found = (pdf.group(0), title); break

    if not found:
        print("NOT FOUND:", topic, "- download it by hand into", PDFS); continue
    pdf_url, title = found
    r = s.get(pdf_url, timeout=120)
    if r.ok and r.content[:4] == b"%PDF":                                # a real PDF starts with %PDF
        open(target, "wb").write(r.content)
        print(f"saved {filename} ({len(r.content)//1024} KB) - {title[:70]}")
    else:
        print("failed:", topic, r.status_code)

print(sorted(os.listdir(PDFS)))

## Part 2 - Read every PDF, one row per page
RAG must cite *"page 41 of the KYC Directions"*, so we keep the page number from the start. SEBI PDFs added by hand (named `SEBI_Short_Title.pdf`) are read too.

In [ ]:
!pip install -q pymupdf rank_bm25 sentence-transformers anthropic
import numpy as np
import pandas as pd
import pymupdf

In [ ]:
def read_pdf(path):
    pages = []
    doc = pymupdf.open(path)                                       # open the PDF like a book
    for number, pg in enumerate(doc, start=1):
        pages.append({"page": number, "text": pg.get_text()})      # pull out the text
    doc.close()
    return pages

rows = []
for f in sorted(x for x in os.listdir(PDFS) if x.lower().endswith(".pdf")):
    regulator = f.split("_")[0].upper()                    # "RBI_KYC_..." -> "RBI"
    title = f[len(regulator) + 1:-4].replace("_", " ")     # -> "KYC Master Direction"
    for p in read_pdf(f"{PDFS}/{f}"):
        rows.append({"file": f, "regulator": regulator, "title": title, **p})

data = pd.DataFrame(rows)                                  # one row per page
print(data.shape)

### ⭐ Quality check
Pages with under 20 words are probably scanned images - the AI could never search them.

In [ ]:
data["words"] = data["text"].str.split().str.len()
data.groupby(["regulator", "title"]).agg(pages=("page", "count"), words=("words", "sum"),
                                         empty_pages=("words", lambda w: (w < 20).sum())).reset_index()

In [ ]:
def clean(text):
    text = text.replace("\u00a0", " ")             # odd invisible spaces -> normal spaces
    text = re.sub(r"-\n(\w)", r"\1", text)         # "regu-\nlation" -> "regulation"
    text = re.sub(r"[ \t]+", " ", text)            # many spaces -> one
    text = re.sub(r"\n{2,}", "\n\n", text)         # many blank lines -> one
    return text.strip()

data["text"] = data["text"].apply(clean)

## Part 3 - Chunking
The AI can't read hundreds of pages per question. We cut text into ~300-word chunks that overlap by 50 words, so a rule cut in half still appears whole in one chunk.

In [ ]:
def make_chunks(text, size=300, overlap=50):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))   # take 300 words
        if start + size >= len(words):                       # end of the page
            break
        start += size - overlap                              # step 250, so 50 words repeat
    return chunks

print(len(make_chunks("w " * 700)), "chunks from 700 words")   # should print 3

In [ ]:
chunk_rows = []
for _, row in data.iterrows():
    for i, piece in enumerate(make_chunks(row["text"])):
        chunk_rows.append({"chunk_id": f"{row['file'][:-4]}_p{row['page']}_c{i}",   # e.g. RBI_KYC_Master_Direction_p41_c0
                           "regulator": row["regulator"], "title": row["title"],
                           "file": row["file"], "page": row["page"], "text": piece})
chunks = pd.DataFrame(chunk_rows)
print(len(chunks), "chunks")
chunks.groupby("title").size()

In [ ]:
chunks.to_parquet(f"{OUT}/chunks.parquet", index=False)      # save (Drive stays connected until the last cell)

## Part 4 - Embeddings: search by meaning
An embedding model turns text into 384 numbers. Similar meaning -> similar numbers, even with different words.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
model = SentenceTransformer("BAAI/bge-small-en-v1.5")       # free, ~130 MB, downloads once
print("Each text becomes", model.get_sentence_embedding_dimension(), "numbers")

### ⭐ See it work on 3 sentences
A and B mean the same with different words; C is another topic. A-B should score clearly higher than A-C.

In [ ]:
examples = ["KYC must be updated periodically for high-risk customers",
            "Customer identity details need re-verification every two years",
            "The payment aggregator must have a net worth of 25 crore"]
vectors = model.encode(examples, normalize_embeddings=True)
pd.DataFrame((vectors @ vectors.T).round(2), index=list("ABC"), columns=list("ABC"))   # cosine similarity

In [ ]:
chunk_vectors = model.encode(chunks["text"].tolist(), normalize_embeddings=True, batch_size=32, show_progress_bar=True)
np.save(f"{OUT}/chunk_vectors.npy", chunk_vectors)
print(chunk_vectors.shape)

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge wants this on questions only

def search(question, k=5):
    q = model.encode([QUERY_PREFIX + question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q                        # similarity to every chunk at once
    best = np.argsort(-scores)[:k]                   # the k highest
    found = chunks.iloc[best][["chunk_id", "title", "page", "text"]].copy()
    found.insert(0, "score", scores[best].round(3))
    return found

questions = ["How often must KYC be updated for high risk customers?",
             "What is the maximum amount that can be loaded in a small PPI?",
             "Within how many days must a bank report a fraud to RBI?"]
search(questions[1], 3)

## Part 5 - Hybrid search + re-ranking
Meaning search finds the right *area*; keyword search (BM25) catches exact terms like "small PPI" or "Rs.10,000". We merge both with Reciprocal Rank Fusion, then a re-ranker reads question + chunk *together* to pick the best 5.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in chunks["text"]])      # keyword index

def keyword_search(question, k=5):
    scores = bm25.get_scores(tokenize(question))
    best = np.argsort(-scores)[:k]
    found = chunks.iloc[best][["chunk_id", "title", "page", "text"]].copy()
    found.insert(0, "score", scores[best].round(2))
    return found

In [ ]:
def hybrid_search(question, k=5, pool=20):
    rrf = {}
    for results in (search(question, pool), keyword_search(question, pool)):
        for rank, cid in enumerate(results["chunk_id"], start=1):
            rrf[cid] = rrf.get(cid, 0) + 1 / (60 + rank)      # Reciprocal Rank Fusion
    best = sorted(rrf, key=rrf.get, reverse=True)[:k]
    found = chunks.set_index("chunk_id").loc[best, ["title", "page", "text"]].reset_index()
    found.insert(0, "rrf", [round(rrf[c], 4) for c in best])
    return found

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")    # small, free (~90 MB)

def retrieve(question, k=5, pool=20):
    candidates = hybrid_search(question, k=pool)
    candidates["rerank"] = reranker.predict([(question, t) for t in candidates["text"]])   # reads both together
    return candidates.sort_values("rerank", ascending=False).head(k).reset_index(drop=True)

### ⭐ Compare the three searches

In [ ]:
for q in questions:
    print("\n" + "=" * 90 + "\nQ:", q)
    for name, res in [("meaning", search(q, 3)), ("keyword", keyword_search(q, 3)), ("FINAL", retrieve(q, 3))]:
        print(f"  {name}:")
        for _, r in res.iterrows():
            print(f"    {r['title']}  p.{r['page']}  | {r['text'][:100]}...")

## Part 6 - Generation with 3 engines
**R**etrieve (Parts 4-5) -> **A**ugment (numbered sources in the prompt) -> **G**enerate (write the answer, citing [n]).
The writer is whichever engine is available, best first - so a demo never fails because of a key:

| Engine | Needs |
|---|---|
| 1. Claude | `ANTHROPIC_API_KEY` in Secrets |
| 2. Free open-source model (Qwen2.5-1.5B) | a GPU runtime |
| 3. Extractive: exact sentences, no AI | nothing |

In [ ]:
SYSTEM = """You answer questions about Indian financial regulations using ONLY the numbered sources provided.

Rules:
1. Every factual sentence must end with its source number in square brackets, e.g. [2].
2. Quote numbers, limits and deadlines exactly as written in the sources.
3. If the sources do not contain the answer, say exactly: "I couldn't find this in the provided RBI/SEBI documents." Do not guess or use outside knowledge.
4. If the sources only partly answer, say what is covered and what is missing.
5. Keep it short: 2-5 sentences. Plain English.
6. This is information, not legal advice."""
REFUSAL = "I couldn't find this in the provided RBI/SEBI documents."

def build_context(found):
    return "\n\n".join(f"[{i}] {r.title}, page {r.page}\n{r.text}" for i, r in enumerate(found.itertuples(), start=1))

print(build_context(retrieve(questions[1]))[:800])      # always look at what the AI receives

### ⭐ Calibrate the "I don't know" threshold
The re-ranker scores how well the best sentence answers. Real questions should score well above 0, off-topic ones far below.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.;:])\s+", text) if 8 <= len(s.split()) <= 80]

def best_sentence_score(question):
    found = retrieve(question)
    sents = [s for t in found["text"] for s in split_sentences(t)]
    return round(float(max(reranker.predict([(question, s) for s in sents]))), 2)

tests = questions + ["What is the current repo rate?"]      # the last one is NOT in our documents
for q in tests:
    print(f"{best_sentence_score(q):>7}   {q}")

In [ ]:
### The 3 answer engines
import anthropic, torch
from google.colab import userdata

THRESHOLD = 0.0                                             # from the scores above

claude = None                                               # Engine 1: Claude, only if a key works
try:
    claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    print("Engine 1: Claude key found")
except Exception:
    print("Engine 1: no Claude key - skipped")
MODEL = "claude-haiku-5-5"

local_llm = None                                            # Engine 2: free model, only on a GPU
if torch.cuda.is_available():
    from transformers import pipeline
    local_llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto", torch_dtype=torch.float16)
    print("Engine 2: free local model ready")
else:
    print("Engine 2: no GPU - skipped")

def extractive_answer(question, found, n=2):                # Engine 3: exact sentences, always works
    candidates = [(i, s) for i, r in enumerate(found.itertuples(), start=1) for s in split_sentences(r.text)]
    if not candidates:
        return REFUSAL
    scores = reranker.predict([(question, s) for _, s in candidates])
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    keep = [(i, s) for score, (i, s) in ranked[:n] if score >= THRESHOLD]
    return " ".join(f"{s} [{i}]" for i, s in keep) if keep else REFUSAL
print("Engine 3: extractive answer ready")

In [ ]:
def generate(question, context, found):
    user_msg = f"Sources:\n\n{context}\n\nQuestion: {question}"
    if claude is not None:
        try:
            r = claude.messages.create(model=MODEL, max_tokens=400, system=SYSTEM,
                                       messages=[{"role": "user", "content": user_msg}])
            return r.content[0].text, "Claude"
        except Exception as e:
            print("  Claude unavailable:", type(e).__name__, "- falling back")
    if local_llm is not None:
        out = local_llm([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}],
                        max_new_tokens=300, do_sample=False)
        return out[0]["generated_text"][-1]["content"], "Free local model"
    return extractive_answer(question, found), "Extractive (no AI)"


def ask(question, k=5):
    found = retrieve(question, k=k)                                        # R
    if found["rerank"].iloc[0] < THRESHOLD:                                # off-topic: refuse, no AI call
        return {"answer": REFUSAL, "citations": [], "engine": "retrieval threshold", "refused": True, "found": found}
    answer, engine = generate(question, build_context(found), found)       # A + G
    used = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 0 < int(n) <= len(found)})
    cites = [f"[{n}] {found.iloc[n-1]['title']}, p.{found.iloc[n-1]['page']}" for n in used]
    return {"answer": answer, "citations": cites, "engine": engine,
            "refused": answer.startswith(REFUSAL[:25]), "found": found}

### ⭐ Test, including a question it must refuse

In [ ]:
for q in tests:
    out = ask(q)
    print("\n" + "=" * 90 + "\nQ:", q)
    print(f"\nA ({out['engine']}):", out["answer"])
    print("\nSources:", *out["citations"], sep="\n  ")

## Part 7 - Evaluation: questions with known answers
For each question we check: **retrieval** (were the right document and text in the top 5?), **answer** (does it contain the expected number?) and **refusal** (does it refuse what the documents cannot answer?).

In [ ]:
eval_set = [
    {"q": "How often must KYC be updated for high-risk customers?", "doc": "KYC", "expect": "two years"},
    {"q": "How often is periodic updation of KYC required for medium risk customers?", "doc": "KYC", "expect": "eight years"},
    {"q": "How often is periodic updation of KYC required for low risk customers?", "doc": "KYC", "expect": "10 years"},
    {"q": "What is the maximum amount that can be loaded in a small PPI in a month?", "doc": "PPI", "expect": "10,000"},
    {"q": "What is the total amount that can be loaded in a small PPI during a financial year?", "doc": "PPI", "expect": "1,20,000"},
    {"q": "What is the cash withdrawal limit per transaction for non-bank PPIs?", "doc": "PPI", "expect": "2,000"},
    {"q": "Within how many days must a bank report theft, burglary or robbery to RBI?", "doc": "Fraud", "expect": "seven days"},
    {"q": "How much time must a bank give a person to reply to a show cause notice before classifying a fraud?", "doc": "Fraud", "expect": "21 days"},
    {"q": "What is the current repo rate?", "doc": None, "expect": None},             # must refuse
    {"q": "What is the GST rate on restaurant bills?", "doc": None, "expect": None},  # must refuse
    {"q": "Who won the IPL final in 2024?", "doc": None, "expect": None},             # must refuse
]
norm = lambda t: re.sub(r"[\s,/\-]", "", str(t).lower())          # "Rs.1,20,000/-" matches "1,20,000"

In [ ]:
rows = []
for e in eval_set:
    out = ask(e["q"])
    if e["expect"] is None:                                           # off-topic question
        rows.append({"question": e["q"], "type": "refuse", "retrieval_hit": None, "correct": out["refused"]})
    else:
        hit = any(e["doc"].lower() in r.title.lower() and norm(e["expect"]) in norm(r.text) for r in out["found"].itertuples())
        rows.append({"question": e["q"], "type": "fact", "retrieval_hit": hit,
                     "correct": (not out["refused"]) and norm(e["expect"]) in norm(out["answer"])})
results = pd.DataFrame(rows)
results

In [ ]:
facts, refuse = results[results["type"] == "fact"], results[results["type"] == "refuse"]
print(f"Retrieval hit rate : {facts['retrieval_hit'].mean():.0%}")
print(f"Answer accuracy    : {facts['correct'].mean():.0%}")
print(f"Refusal accuracy   : {refuse['correct'].mean():.0%}")
results.to_csv(f"{OUT}/eval_results.csv", index=False)

## Part 8 - The AI agent: Claude chooses tools in a loop
RAG searches once and writes once. An **agent** decides what to do next: search again, filter by regulator, open a chunk, then answer.

```
question -> Claude thinks -> calls a tool -> reads the result -> calls another tool ... -> final answer with citations
```
This part needs the Claude key (engine 1). Without it, the cell explains and skips.

In [ ]:
AGENT_SYSTEM = """You are a compliance research assistant for Indian financial regulations (RBI and SEBI).
Only use facts returned by your tools. Plan first; comparisons need one search per part.
Cite every fact with the chunk id in square brackets, e.g. [RBI_KYC_Master_Direction_p41_c0].
If the tools do not contain the answer, say exactly: "I couldn't find this in the provided RBI/SEBI documents." """

TOOLS = [
    {"name": "search_regulations", "description": "Search the RBI/SEBI documents. Returns 5 chunks with chunk_id, title, page and text.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "read_chunk", "description": "Read one chunk in full by its chunk_id.",
     "input_schema": {"type": "object", "properties": {"chunk_id": {"type": "string"}}, "required": ["chunk_id"]}},
]

def run_tool(name, args):
    if name == "search_regulations":
        found = retrieve(args["query"])
        return found[["chunk_id", "title", "page", "text"]].to_json(orient="records")
    if name == "read_chunk":
        row = chunks[chunks["chunk_id"] == args["chunk_id"]]
        return row.to_json(orient="records") if len(row) else "No chunk with that id."
    return "Unknown tool"

In [ ]:
def run_agent(question, max_steps=6):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = claude.messages.create(model=MODEL, max_tokens=800, system=AGENT_SYSTEM, tools=TOOLS, messages=messages)
        if r.stop_reason != "tool_use":                                   # ready to answer
            return " ".join(b.text for b in r.content if b.type == "text")
        messages.append({"role": "assistant", "content": r.content})     # remember what it asked for
        results = []
        for block in r.content:
            if block.type == "tool_use":
                print(f"  step {step}: {block.name}({block.input})")       # watch the agent think
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": run_tool(block.name, block.input)})
        messages.append({"role": "user", "content": results})            # give it the results
    return "Stopped after too many steps."

if claude is None:
    print("Agent needs the Claude key (engine 1) - add ANTHROPIC_API_KEY in Secrets to try it.")
else:
    print(run_agent("Compare how often KYC must be updated for high, medium and low risk customers."))

## Part 9 - Save for the website
The website reads `chunks.json`. (The GitHub repo rebuilds this automatically every Monday with the same steps.)

In [ ]:
chunks.rename(columns={"chunk_id": "id"})[["id", "regulator", "title", "page", "text"]].to_json(
    f"{OUT}/chunks.json", orient="records", force_ascii=False)
print("Saved", len(chunks), "chunks for the website")

In [ ]:
drive.flush_and_unmount()       # always the LAST cell